In [2]:
# ── Cell 1 — imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
print("imports done")

imports done


In [3]:
# ── Cell 2 — load week 1 outputs ─────────────────────────────────────
demand_df  = pd.read_csv(r'F:\projectss\personal\blinkit_project\data\processed\demand_table.csv')
pincode_df = pd.read_csv(r'F:\projectss\personal\blinkit_project\data\5c2f62fe-5afa-4119-a499-fec9d604d5bd.csv')

print("demand_table shape :", demand_df.shape)
print("columns            :", demand_df.columns.tolist())
print(demand_df.head(3))

demand_table shape : (81898, 9)
columns            : ['product_id', 'product_name', 'pincode', 'area_name', 'latitude', 'longitude', 'order_dow', 'order_hour_of_day', 'demand']
   product_id   product_name  pincode area_name  latitude  longitude  \
0        4605  Yellow Onions   382210    Bakrol     23.03      72.63   
1        4605  Yellow Onions   382210    Bakrol     23.03      72.63   
2        4605  Yellow Onions   382210    Bakrol     23.03      72.63   

   order_dow  order_hour_of_day  demand  
0          0                  0      10  
1          0                  1       6  
2          0                  2       4  


In [4]:
# ── Cell 3 — get unique product+pincode combos ────────────────────────
combos = demand_df[
    ['product_id','product_name','pincode',
     'area_name','latitude','longitude']
].drop_duplicates().reset_index(drop=True)

print(f"Unique product+pincode combos: {len(combos)}")
print(combos.head(5))

Unique product+pincode combos: 500
   product_id   product_name  pincode  area_name  latitude  longitude
0        4605  Yellow Onions   382210     Bakrol     23.03      72.63
1        4605  Yellow Onions   382220     Adroda     22.83      72.31
2        4605  Yellow Onions   382225     Vautha     22.67      72.54
3        4605  Yellow Onions   382260  Chandisar     22.80      72.49
4        4605  Yellow Onions   382330    Pardhol     23.10      72.73


In [5]:
# ── Cell 4 — build 26 weeks of hourly rows with weekly noise ──────────
# THIS IS THE KEY FIX
# each week gets ±20% random noise on demand
# so lag_168 (same hour last week) is genuinely different from today

NUM_WEEKS  = 26
START_DATE = '2024-01-01'

all_rows = []

for week in range(NUM_WEEKS):
    week_start    = pd.Timestamp(START_DATE) + pd.Timedelta(weeks=week)
    noise_factor  = np.random.uniform(0.80, 1.20)  # different per week

    for _, combo in combos.iterrows():
        base = demand_df[
            (demand_df['product_id'] == combo['product_id']) &
            (demand_df['pincode']    == combo['pincode'])
        ][['order_dow','order_hour_of_day','demand']].copy()

        if base.empty:
            continue

        # apply weekly noise
        base['demand'] = (base['demand'] * noise_factor).clip(lower=0).round(0)

        for _, row in base.iterrows():
            dt = week_start + pd.Timedelta(
                days=int(row['order_dow']),
                hours=int(row['order_hour_of_day'])
            )
            all_rows.append({
                'datetime'         : dt,
                'week_number'      : week + 1,
                'product_id'       : combo['product_id'],
                'product_name'     : combo['product_name'],
                'pincode'          : combo['pincode'],
                'area_name'        : combo['area_name'],
                'latitude'         : combo['latitude'],
                'longitude'        : combo['longitude'],
                'order_dow'        : int(row['order_dow']),
                'order_hour_of_day': int(row['order_hour_of_day']),
                'demand'           : float(row['demand']),
            })

ts_df = pd.DataFrame(all_rows)
ts_df = ts_df.sort_values(
    ['product_id','pincode','datetime']
).reset_index(drop=True)

print(f"Time series shape : {ts_df.shape}")
print(f"Date range        : {ts_df['datetime'].min()} → {ts_df['datetime'].max()}")
print(ts_df[['datetime','product_id','pincode','demand']].head(8))

Time series shape : (2129348, 11)
Date range        : 2024-01-01 00:00:00 → 2024-06-30 23:00:00
             datetime  product_id  pincode  demand
0 2024-01-01 00:00:00        4605   382210    9.00
1 2024-01-01 01:00:00        4605   382210    6.00
2 2024-01-01 02:00:00        4605   382210    4.00
3 2024-01-01 03:00:00        4605   382210    2.00
4 2024-01-01 04:00:00        4605   382210    1.00
5 2024-01-01 05:00:00        4605   382210    3.00
6 2024-01-01 06:00:00        4605   382210    9.00
7 2024-01-01 07:00:00        4605   382210   38.00


In [6]:
# ── Cell 5 — verify no leakage before adding features ─────────────────
# demand should vary week to week for same product+pincode+dow+hour

sample = ts_df[
    (ts_df['product_id'] == ts_df['product_id'].iloc[0]) &
    (ts_df['pincode']    == ts_df['pincode'].iloc[0])
].head(20)

print("Same product+pincode across weeks — demand should vary:")
print(sample[['datetime','order_dow','order_hour_of_day','demand']])
print(f"\nDemand std (should be > 0): {sample['demand'].std():.2f}")

Same product+pincode across weeks — demand should vary:
              datetime  order_dow  order_hour_of_day  demand
0  2024-01-01 00:00:00          0                  0    9.00
1  2024-01-01 01:00:00          0                  1    6.00
2  2024-01-01 02:00:00          0                  2    4.00
3  2024-01-01 03:00:00          0                  3    2.00
4  2024-01-01 04:00:00          0                  4    1.00
5  2024-01-01 05:00:00          0                  5    3.00
6  2024-01-01 06:00:00          0                  6    9.00
7  2024-01-01 07:00:00          0                  7   38.00
8  2024-01-01 08:00:00          0                  8   78.00
9  2024-01-01 09:00:00          0                  9  120.00
10 2024-01-01 10:00:00          0                 10  149.00
11 2024-01-01 11:00:00          0                 11  177.00
12 2024-01-01 12:00:00          0                 12  170.00
13 2024-01-01 13:00:00          0                 13  154.00
14 2024-01-01 14:00:00       

In [7]:
demand_df.duplicated(subset=['product_id','pincode','order_dow','order_hour_of_day']).sum()  

0

In [8]:
# ── Cell 6 — cyclical time features ───────────────────────────────────
ts_df['hour_sin']  = np.sin(2 * np.pi * ts_df['order_hour_of_day'] / 24)
ts_df['hour_cos']  = np.cos(2 * np.pi * ts_df['order_hour_of_day'] / 24)
ts_df['dow_sin']   = np.sin(2 * np.pi * ts_df['order_dow'] / 7)
ts_df['dow_cos']   = np.cos(2 * np.pi * ts_df['order_dow'] / 7)
ts_df['week_sin']  = np.sin(2 * np.pi * ts_df['week_number'] / 52)
ts_df['week_cos']  = np.cos(2 * np.pi * ts_df['week_number'] / 52)
ts_df['month']     = ts_df['datetime'].dt.month

print("Cyclical features added")
print(ts_df[['order_hour_of_day','hour_sin','hour_cos']].head(5))

Cyclical features added
   order_hour_of_day  hour_sin  hour_cos
0                  0      0.00      1.00
1                  1      0.26      0.97
2                  2      0.50      0.87
3                  3      0.71      0.71
4                  4      0.87      0.50


In [9]:
# ── Cell 7 — calendar features ────────────────────────────────────────
ts_df['is_weekend']   = ts_df['order_dow'].isin([5, 6]).astype(int)
ts_df['is_morning']   = ts_df['order_hour_of_day'].between(6,  11).astype(int)
ts_df['is_afternoon'] = ts_df['order_hour_of_day'].between(12, 17).astype(int)
ts_df['is_evening']   = ts_df['order_hour_of_day'].between(18, 22).astype(int)
ts_df['is_night']     = ts_df['order_hour_of_day'].isin(
                            [23,0,1,2,3,4,5]).astype(int)

festival_dates = [
    '2024-01-14','2024-01-26','2024-03-25',
    '2024-04-14','2024-08-15','2024-10-02',
    '2024-11-01','2024-11-15'
]
ts_df['date_str']    = ts_df['datetime'].dt.strftime('%Y-%m-%d')
ts_df['is_festival'] = ts_df['date_str'].isin(festival_dates).astype(int)

print("Calendar features added")
print(f"Weekend rows  : {ts_df['is_weekend'].sum():,}")
print(f"Festival rows : {ts_df['is_festival'].sum():,}")

Calendar features added
Weekend rows  : 609,206
Festival rows : 46,820


In [10]:
# ── Cell 8 — lag features (leak-free) ────────────────────────────────
grp = ts_df.groupby(['product_id','pincode'])['demand']

ts_df['lag_1']   = grp.shift(1)    # 1 hour ago
ts_df['lag_24']  = grp.shift(24)   # same hour yesterday
ts_df['lag_48']  = grp.shift(48)   # same hour 2 days ago
ts_df['lag_168'] = grp.shift(168)  # same hour last week

# LEAKAGE CHECK — must be close to 0%
exact = (ts_df['lag_168'] == ts_df['demand']).mean()
print(f"Leakage check — lag_168 == demand: {exact*100:.2f}%")
print("Must be near 0% — if so we are good!\n")

print(ts_df[['datetime','demand','lag_1','lag_24','lag_168']].head(10))

Leakage check — lag_168 == demand: 3.79%
Must be near 0% — if so we are good!

             datetime  demand  lag_1  lag_24  lag_168
0 2024-01-01 00:00:00    9.00    NaN     NaN      NaN
1 2024-01-01 01:00:00    6.00   9.00     NaN      NaN
2 2024-01-01 02:00:00    4.00   6.00     NaN      NaN
3 2024-01-01 03:00:00    2.00   4.00     NaN      NaN
4 2024-01-01 04:00:00    1.00   2.00     NaN      NaN
5 2024-01-01 05:00:00    3.00   1.00     NaN      NaN
6 2024-01-01 06:00:00    9.00   3.00     NaN      NaN
7 2024-01-01 07:00:00   38.00   9.00     NaN      NaN
8 2024-01-01 08:00:00   78.00  38.00     NaN      NaN
9 2024-01-01 09:00:00  120.00  78.00     NaN      NaN


In [11]:
# ── Cell 9 — rolling features (leak-free) ────────────────────────────
# shift(1) inside rolling ensures current row never leaks into window

grp = ts_df.groupby(['product_id','pincode'])['demand']

ts_df['rolling_mean_24']  = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=1).mean())
ts_df['rolling_mean_168'] = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=1).mean())
ts_df['rolling_mean_720'] = grp.transform(
    lambda x: x.shift(1).rolling(720, min_periods=1).mean())
ts_df['rolling_std_24']   = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=2).std().fillna(0))
ts_df['rolling_std_168']  = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=2).std().fillna(0))
ts_df['rolling_max_24']   = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=1).max())
ts_df['rolling_max_168']  = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=1).max())
ts_df['ewma_24']  = grp.transform(
    lambda x: x.shift(1).ewm(span=24,  adjust=False).mean())
ts_df['ewma_168'] = grp.transform(
    lambda x: x.shift(1).ewm(span=168, adjust=False).mean())

print("Rolling features added")
print(ts_df[['datetime','demand',
             'rolling_mean_24','rolling_mean_168','ewma_24']].head(8))

Rolling features added
             datetime  demand  rolling_mean_24  rolling_mean_168  ewma_24
0 2024-01-01 00:00:00    9.00              NaN               NaN      NaN
1 2024-01-01 01:00:00    6.00             9.00              9.00     9.00
2 2024-01-01 02:00:00    4.00             7.50              7.50     8.76
3 2024-01-01 03:00:00    2.00             6.33              6.33     8.38
4 2024-01-01 04:00:00    1.00             5.25              5.25     7.87
5 2024-01-01 05:00:00    3.00             4.40              4.40     7.32
6 2024-01-01 06:00:00    9.00             4.17              4.17     6.97
7 2024-01-01 07:00:00   38.00             4.86              4.86     7.14


In [13]:
# ── Cell 10 — fetch weather (free, no api key) ────────────────────────
def fetch_weather(lat, lon, start='2024-01-01', end='2024-06-30'):
    url    = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude"  : lat,
        "longitude" : lon,
        "start_date": start,
        "end_date"  : end,
        "hourly"    : "temperature_2m,precipitation,relativehumidity_2m",
        "timezone"  : "Asia/Kolkata"
    }
    data = requests.get(url, params=params).json()
    return pd.DataFrame({
        'datetime'   : pd.to_datetime(data['hourly']['time']),
        'temperature': data['hourly']['temperature_2m'],
        'rainfall'   : data['hourly']['precipitation'],
        'humidity'   : data['hourly']['relativehumidity_2m'],
    })

print("Fetching Ahmedabad weather 2024...")
weather_df = fetch_weather(23.0225, 72.5714)
weather_df['is_raining'] = (weather_df['rainfall'] > 0.5).astype(int)
weather_df['is_hot']     = (weather_df['temperature'] > 35).astype(int)

print(f"Weather shape: {weather_df.shape}")
print(weather_df.head(5))

weather_df.to_csv(r'F:\projectss\personal\blinkit_project\data\external/ahmedabad_weather_2024.csv', index=False)
print("Weather saved!")

Fetching Ahmedabad weather 2024...
Weather shape: (4368, 6)
             datetime  temperature  rainfall  humidity  is_raining  is_hot
0 2024-01-01 00:00:00        17.50      0.00        80           0       0
1 2024-01-01 01:00:00        16.80      0.00        84           0       0
2 2024-01-01 02:00:00        16.00      0.00        88           0       0
3 2024-01-01 03:00:00        15.60      0.00        88           0       0
4 2024-01-01 04:00:00        15.50      0.00        88           0       0
Weather saved!


In [15]:
weather_df.duplicated(subset=['datetime']).sum()

0

In [14]:
# ── Cell 11 — merge weather ───────────────────────────────────────────
ts_df['datetime']      = pd.to_datetime(ts_df['datetime'])
weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])

ts_df = ts_df.merge(
    weather_df[['datetime','temperature','rainfall',
                'humidity','is_raining','is_hot']],
    on='datetime', how='left'
)

# fill missing weather with sensible defaults
ts_df['temperature'] = ts_df['temperature'].fillna(ts_df['temperature'].median())
ts_df['rainfall']    = ts_df['rainfall'].fillna(0)
ts_df['humidity']    = ts_df['humidity'].fillna(ts_df['humidity'].median())
ts_df['is_raining']  = ts_df['is_raining'].fillna(0)
ts_df['is_hot']      = ts_df['is_hot'].fillna(0)

print("Weather merged")
print(f"Missing temp values: {ts_df['temperature'].isnull().sum()}")
print(ts_df[['datetime','temperature','rainfall','is_raining']].head(5))

Weather merged
Missing temp values: 0
             datetime  temperature  rainfall  is_raining
0 2024-01-01 00:00:00        17.50      0.00           0
1 2024-01-01 01:00:00        16.80      0.00           0
2 2024-01-01 02:00:00        16.00      0.00           0
3 2024-01-01 03:00:00        15.60      0.00           0
4 2024-01-01 04:00:00        15.50      0.00           0


In [16]:
# ── Cell 12 — drop nulls from lag features ────────────────────────────
print(f"Shape before dropna: {ts_df.shape}")

ts_df = ts_df.dropna(subset=[
    'lag_1','lag_24','lag_168',
    'rolling_mean_24','rolling_mean_168'
])

print(f"Shape after dropna : {ts_df.shape}")
print(f"Null count total   : {ts_df.isnull().sum().sum()}")

Shape before dropna: (2129348, 43)
Shape after dropna : (2045348, 43)
Null count total   : 0


In [18]:
# ── Cell 13 — final feature list and save ─────────────────────────────
FEATURE_COLS = [
    'hour_sin','hour_cos','dow_sin','dow_cos',
    'week_sin','week_cos','month',
    'is_weekend','is_morning','is_afternoon',
    'is_evening','is_night','is_festival',
    'lag_1','lag_24','lag_48','lag_168',
    'rolling_mean_24','rolling_mean_168','rolling_mean_720',
    'rolling_std_24','rolling_std_168',
    'rolling_max_24','rolling_max_168',
    'ewma_24','ewma_168',
    'temperature','rainfall','humidity',
    'is_raining','is_hot',
]

print(f"Total features : {len(FEATURE_COLS)}")
print(f"X shape        : {ts_df[FEATURE_COLS].shape}")
print(f"y shape        : {ts_df['demand'].shape}")
print(f"\nDemand stats:\n{ts_df['demand'].describe()}")

ts_df.to_csv(r'F:\projectss\personal\blinkit_project\data\processed\features_df_fixed.csv', index=False)
print("\nSaved → data/processed/features_df_fixed.csv")
print("Notebook 02 complete!")

Total features : 31
X shape        : (2045348, 31)
y shape        : (2045348,)

Demand stats:
count   2045348.00
mean         64.53
std          83.14
min           1.00
25%          11.00
50%          44.00
75%          83.00
max        1183.00
Name: demand, dtype: float64

Saved → data/processed/features_df_fixed.csv
Notebook 02 complete!


In [20]:
import pandas as pd
ts_df = pd.read_csv(r'F:\projectss\personal\blinkit_project\data\processed\features_df_fixed.csv')
print("Shape:", ts_df.shape)
print("Leakage check:", (ts_df['lag_168'] == ts_df['demand']).mean() * 100, "%")
print("Demand stats:\n", ts_df['demand'].describe())

Shape: (2045348, 43)
Leakage check: 3.9472989437494257 %
Demand stats:
 count   2045348.00
mean         64.53
std          83.14
min           1.00
25%          11.00
50%          44.00
75%          83.00
max        1183.00
Name: demand, dtype: float64
